# 4.2 - Analytics - Geospatial Diagnostic and Network Mapping

Este notebook gera diagnósticos espaciais e mapas temáticos de alta resolução (Padrão Q1) da distribuição das estações aprovadas (HQ) e reprovadas (LQ) pelo Controle de Qualidade.

In [ ]:
%config Completer.use_jedi = False
%load_ext autoreload
%autoreload 2

import sys, os, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd() if Path.cwd().name == 'src' else Path.cwd() / 'src'
sys.path.insert(0, str(PROJECT_ROOT.parent))

from src.config import ROOT
from src.visualization.plots import plot_station_map
from src.data.loader import load_gauge_info
from src.utils.logger import save_figure

RESULTS_DIR = ROOT / 'src' / '5 - Results'
FIGURES_DIR = ROOT / 'src' / '4 - Figures'

## 1. Carregamento dos Dados de QC e Metadados Geográficos

In [ ]:
df_qc = pd.read_parquet(RESULTS_DIR / 'qc_classification_results.parquet')
df_info = load_gauge_info()
merged = pd.merge(df_info, df_qc[['gauge_code', 'hybrid_quality']].drop_duplicates(), on='gauge_code', how='inner')
print(f'Estações georreferenciadas com classificação de qualidade: {len(merged):,}')
merged.head()

## 2. Mapa Espacial das Estações por Qualidade (HQ vs LQ)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
hq_pts = merged[merged['hybrid_quality'] == 'HQ']
lq_pts = merged[merged['hybrid_quality'] == 'LQ']

ax.scatter(hq_pts['long'], hq_pts['lat'], c='dodgerblue', s=12, label='Alta Qualidade (HQ)', alpha=0.7)
ax.scatter(lq_pts['long'], lq_pts['lat'], c='crimson', s=14, label='Baixa Qualidade (LQ)', marker='x', alpha=0.8)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Rede Pluviométrica — Diagnóstico de Controle de Qualidade (QC-VORONOI)')
ax.legend(loc='lower left', markerscale=2)
ax.grid(True, linestyle=':', alpha=0.6)

save_figure(fig, 'qc_spatial_diagnostic_map.png', subdir='4 - Figures')
plt.show()